# QIS portfolio correlation and diversification review

Set the inputs and parameters below, then **Run All**. The default CSVs are simulated. Calculations live in the Python package; this notebook displays the same report exported to HTML.

In [ ]:
from pathlib import Path

from IPython.display import HTML, display

from qis_risk.data import DEMO_UNIVERSE, InputValidationError, load_inputs
from qis_risk.model import ModelConfig, PortfolioConfig, run_review
from qis_risk.report import DisplayConfig, export_html, render_input_error, render_report

## Inputs and parameters

Weights are decimal fractions (`0.25 = 25%`). CSV rows declare the trading calendar; retain missing strategy observations as blanks. Use an explicit ordered universe for real inputs. The book holds fixed index units between rebalance closes.

In [ ]:
LEVELS_CSV = Path("data/demo/index_levels.csv")
WEIGHTS_CSV = Path("data/demo/rebalance_weights.csv")
UNIVERSE = DEMO_UNIVERSE
AS_OF_DATE = None
CALENDAR_COMPLETE_THROUGH = None
REPORT_PATH = Path("reports/qis_risk_dashboard.html")

PORTFOLIO = PortfolioConfig(
    portfolio_id="QIS demonstration portfolio",
    currency="USD",
    return_basis="excess_return",
    valuation_close="Common simulated weekday close",
    live_backtested="simulated",
    data_vintage="Deterministic simulation, seed 42",
)
MODEL = ModelConfig()
PM_CONTROLS = DisplayConfig(history_months=36, top_n_pairs=5, show_heatmap=True)

## Run the review

Model defaults: 60-day volatility half-life, 126/42-day slow/fast correlation half-lives, 60-observation standardization warmup, 504 common raw observations and 444 standardized vectors before publishing risk. Change numerical settings with `ModelConfig(...)`; display controls only affect presentation.

In [ ]:
try:
    inputs = load_inputs(
        LEVELS_CSV, WEIGHTS_CSV, universe=UNIVERSE,
        calendar_complete_through=CALENDAR_COMPLETE_THROUGH,
    )
    result = run_review(inputs, PORTFOLIO, MODEL, as_of_date=AS_OF_DATE)
    report_html = render_report(result, PM_CONTROLS)
except (InputValidationError, OSError) as error:
    report_html = render_input_error(error)

In [ ]:
display(HTML(report_html))

## Offline HTML

The export contains the exact report shown above, including embedded charts and the expandable appendix. Generated reports and real inputs are ignored by Git.

In [ ]:
export_html(report_html, REPORT_PATH)